In [ ]:
# Task 1: Download and Load Data

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Download diabetes dataset
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/refs/heads/master/diabetes.csv"
df = pd.read_csv(url)

# Separate features (X) and target (y)
X = df.iloc[:, :-1].values  # First 8 columns: features
y = df.iloc[:, -1].values    # Last column: diabetes (0 or 1)

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")


In [ ]:
# Task 2: Neural Network Implementation

class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Architecture: 8 → 6 → 4 → 2
        self.backbone = nn.Sequential(
            nn.Linear(in_features=8, out_features=6),
            nn.ReLU(),
            nn.Linear(in_features=6, out_features=4),
            nn.ReLU(),
            nn.Linear(in_features=4, out_features=2),
        )
    
    def forward(self, x):
        return self.backbone(x)

model = DiabetesNet()
print(model)


In [ ]:
# Task 3: Neural Network Training

from torch.utils.data import TensorDataset, DataLoader

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)  # CrossEntropyLoss requires Long type
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Create DataLoaders for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Loss function and optimizer
loss_func = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

# Training configuration
total_epochs = 150

# Metrics storage
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

print("Starting training...")


In [ ]:
# Test function
def test(model, test_loader, loss_func):
    model.eval()
    losses = []
    preds = []
    gt = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            logits = model(inputs)
            loss = loss_func(logits, labels)
            losses.append(loss.item())
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.numpy())
            gt.extend(labels.numpy())
    
    test_acc = accuracy_score(gt, preds)
    test_loss = sum(losses) / len(losses)
    return test_loss, test_acc


In [ ]:
# Training loop
for epoch in range(total_epochs):
    model.train()
    losses = []
    preds = []
    gt = []
    
    for inputs, labels in train_loader:
        logits = model(inputs)
        loss = loss_func(logits, labels)
        
        # Backpropagation
        loss.backward()
        optim.step()
        optim.zero_grad()
        
        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.detach().numpy())
        gt.extend(labels.detach().numpy())
    
    train_acc = accuracy_score(gt, preds)
    train_loss = sum(losses) / len(losses)
    all_train_acc.append(train_acc)
    all_train_losses.append(train_loss)
    
    # Test evaluation
    test_loss, test_acc = test(model, test_loader, loss_func)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/{total_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print(f"\nFinal Test Accuracy: {max(all_test_acc)*100:.2f}%")


In [ ]:
# Visualize training results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_train_losses, label='Train Loss', color='orange')
plt.plot(all_test_losses, label='Test Loss', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Test Loss')

plt.subplot(1, 2, 2)
plt.plot(all_train_acc, label='Train Accuracy', color='orange')
plt.plot(all_test_acc, label='Test Accuracy', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Test Accuracy')

plt.tight_layout()
plt.show()


In [ ]:
# Task 4: Optimize Neural Network
# Experiment with different architectures and dropout

class DiabetesNetOptimized(nn.Module):
    def __init__(self):
        super().__init__()
        # Architecture: 8 → 12 → 8 → 2 with dropout regularization
        self.backbone = nn.Sequential(
            nn.Linear(in_features=8, out_features=12),
            nn.ReLU(),
            nn.Dropout(0.2),  # Dropout to prevent overfitting
            nn.Linear(in_features=12, out_features=8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features=8, out_features=2),
        )
    
    def forward(self, x):
        return self.backbone(x)

model_optimized = DiabetesNetOptimized()
print(model_optimized)


In [ ]:
# Train optimized model
optim_optimized = torch.optim.Adam(model_optimized.parameters(), lr=0.01)

all_train_losses_opt = []
all_train_acc_opt = []
all_test_losses_opt = []
all_test_acc_opt = []

print("Training optimized model...")

for epoch in range(total_epochs):
    model_optimized.train()
    losses = []
    preds = []
    gt = []
    
    for inputs, labels in train_loader:
        logits = model_optimized(inputs)
        loss = loss_func(logits, labels)
        
        loss.backward()
        optim_optimized.step()
        optim_optimized.zero_grad()
        
        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.detach().numpy())
        gt.extend(labels.detach().numpy())
    
    train_acc = accuracy_score(gt, preds)
    train_loss = sum(losses) / len(losses)
    all_train_acc_opt.append(train_acc)
    all_train_losses_opt.append(train_loss)
    
    test_loss, test_acc = test(model_optimized, test_loader, loss_func)
    all_test_losses_opt.append(test_loss)
    all_test_acc_opt.append(test_acc)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/{total_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print(f"\nOptimized Model - Final Test Accuracy: {max(all_test_acc_opt)*100:.2f}%")


In [ ]:
# Compare original vs optimized model
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_test_losses, label='Original Model', color='blue')
plt.plot(all_test_losses_opt, label='Optimized Model', color='green')
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('Test Loss Comparison')

plt.subplot(1, 2, 2)
plt.plot(all_test_acc, label='Original Model', color='blue')
plt.plot(all_test_acc_opt, label='Optimized Model', color='green')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend()
plt.title('Test Accuracy Comparison')

plt.tight_layout()
plt.show()

print(f"Original Model Best Accuracy: {max(all_test_acc)*100:.2f}%")
print(f"Optimized Model Best Accuracy: {max(all_test_acc_opt)*100:.2f}%")


In [ ]:
# Task 5: Normalize Data
# Standardize features to improve training stability and convergence

from sklearn.preprocessing import StandardScaler

# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)  # Use same transformation

# Create new tensors and dataloaders with normalized data
X_train_norm_tensor = torch.FloatTensor(X_train_normalized)
X_test_norm_tensor = torch.FloatTensor(X_test_normalized)

train_dataset_norm = TensorDataset(X_train_norm_tensor, y_train_tensor)
test_dataset_norm = TensorDataset(X_test_norm_tensor, y_test_tensor)
train_loader_norm = DataLoader(train_dataset_norm, batch_size=32, shuffle=True)
test_loader_norm = DataLoader(test_dataset_norm, batch_size=32, shuffle=False)

print("Data normalized using StandardScaler")
print(f"Mean before normalization: {X_train.mean(axis=0)[:3]}")
print(f"Mean after normalization: {X_train_normalized.mean(axis=0)[:3]}")
print(f"Std before normalization: {X_train.std(axis=0)[:3]}")
print(f"Std after normalization: {X_train_normalized.std(axis=0)[:3]}")


In [ ]:
# Train model with normalized data
model_normalized = DiabetesNetOptimized()
optim_normalized = torch.optim.Adam(model_normalized.parameters(), lr=0.01)

all_train_losses_norm = []
all_train_acc_norm = []
all_test_losses_norm = []
all_test_acc_norm = []

print("Training with normalized data...")

for epoch in range(total_epochs):
    model_normalized.train()
    losses = []
    preds = []
    gt = []
    
    for inputs, labels in train_loader_norm:
        logits = model_normalized(inputs)
        loss = loss_func(logits, labels)
        
        loss.backward()
        optim_normalized.step()
        optim_normalized.zero_grad()
        
        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.detach().numpy())
        gt.extend(labels.detach().numpy())
    
    train_acc = accuracy_score(gt, preds)
    train_loss = sum(losses) / len(losses)
    all_train_acc_norm.append(train_acc)
    all_train_losses_norm.append(train_loss)
    
    test_loss, test_acc = test(model_normalized, test_loader_norm, loss_func)
    all_test_losses_norm.append(test_loss)
    all_test_acc_norm.append(test_acc)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/{total_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print(f"\nNormalized Data Model - Final Test Accuracy: {max(all_test_acc_norm)*100:.2f}%")


In [ ]:
# Task 6: Model Evaluation on Specific Patient
# Test patient: [2, 197, 70, 45, 543, 30.5, 0.158, 53]
# Ground truth: This patient HAS diabetes

new_patient = np.array([[2, 197, 70, 45, 543, 30.5, 0.158, 53]])

# Normalize the patient data using the same scaler
new_patient_normalized = scaler.transform(new_patient)

# Convert to tensor
patient_tensor = torch.FloatTensor(new_patient_normalized)

# Make prediction with normalized model
model_normalized.eval()
with torch.no_grad():
    pred_logits = model_normalized(patient_tensor)
    pred_probs = torch.softmax(pred_logits, dim=1)
    pred_class = torch.argmax(pred_logits, dim=1)

print("=" * 60)
print("PATIENT EVALUATION")
print("=" * 60)
print(f"Patient data: {new_patient[0]}")
print(f"\nPrediction probabilities:")
print(f"  No Diabetes: {pred_probs[0][0].item():.4f} ({pred_probs[0][0].item()*100:.2f}%)")
print(f"  Has Diabetes: {pred_probs[0][1].item():.4f} ({pred_probs[0][1].item()*100:.2f}%)")
print(f"\nPredicted class: {pred_class.item()}")

if pred_class.item() == 1:
    print("🔴 The patient is predicted to HAVE DIABETES.")
else:
    print("🟢 The patient is predicted to be HEALTHY.")

print(f"\nGround truth: Patient HAS diabetes")
print(f"Prediction correct: {pred_class.item() == 1}")
print("=" * 60)


In [ ]:
# Summary: Compare all model variants

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"1. Original Model (6→4→2):           {max(all_test_acc)*100:.2f}%")
print(f"2. Optimized Model (12→8→2+Dropout): {max(all_test_acc_opt)*100:.2f}%")
print(f"3. Normalized Data Model:            {max(all_test_acc_norm)*100:.2f}%")
print("=" * 60)

# Plot final comparison
plt.figure(figsize=(10, 5))
plt.plot(all_test_acc, label='Original (6→4→2)', alpha=0.7)
plt.plot(all_test_acc_opt, label='Optimized (12→8→2+Dropout)', alpha=0.7)
plt.plot(all_test_acc_norm, label='Normalized Data', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('Model Performance Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
